# Tableau RAG Bot - Groq + Qdrant + Streamlit
Pure RAG implementation (No LangChain, No Docling)


In [ ]:
!pip install -q qdrant-client sentence-transformers PyMuPDF groq streamlit python-dotenv


In [ ]:
import fitz, uuid, os
from google.colab import files
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from groq import Groq


In [ ]:
import os

# Read the Groq API key from a secret rather than hardcoding it.
# - In Google Colab: Tools -> Secrets -> add a secret named GROQ_API_KEY,
#   toggle "Notebook access" on, then run this cell.
# - In local Jupyter / VS Code: export GROQ_API_KEY in your shell before
#   launching, or set it via os.environ in an earlier cell.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ["GROQ_API_KEY"]

groq_client = Groq(api_key=GROQ_API_KEY)


## Upload Tableau PDFs

In [ ]:
uploaded = files.upload()


Saving tableau_desktop.pdf to tableau_desktop.pdf


In [ ]:
def extract_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages=[]
    for p in range(len(doc)):
        text = doc[p].get_text()
        if text.strip():
            pages.append({'page':p+1,'text':text})
    return pages

def chunk_text(text, chunk_size=500, overlap=100):
    words=text.split()
    chunks=[]
    start=0
    while start < len(words):
        chunks.append(' '.join(words[start:start+chunk_size]))
        start += chunk_size-overlap
    return chunks


In [ ]:
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
vector_size = embedding_model.get_sentence_embedding_dimension()
client = QdrantClient(
    path="./qdrant_data"
)
COLLECTION_NAME='tableau_docs'
client.recreate_collection(collection_name=COLLECTION_NAME, vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_23288/2070634182.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  vector_size = embedding_model.get_sentence_embedding_dimension()


RuntimeError: Storage folder ./qdrant_data is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

In [ ]:
points=[]
for pdf_file in uploaded.keys():
    for page in extract_pdf(pdf_file):
        for chunk in chunk_text(page['text']):
            vec = embedding_model.encode(chunk).tolist()
            points.append(PointStruct(id=str(uuid.uuid4()), vector=vec, payload={'source':pdf_file,'page':page['page'],'content':chunk}))
client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f'Indexed {len(points)} chunks')


Indexed 3780 chunks


In [ ]:
def retrieve(query, top_k=5):

    qv = embedding_model.encode(query).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=qv,
        limit=top_k
    )

    return results.points

In [ ]:
results = retrieve(
    "How do I create a calculated field?"
)

for r in results:

    print("=" * 80)

    print("Score:", r.score)

    print(
        "Source:",
        r.payload["source"]
    )

    print(
        "Page:",
        r.payload["page"]
    )

    print(
        r.payload["content"][:500]
    )

Score: 0.8391958269501807
Source: tableau_desktop.pdf
Page: 3018
of each view in the dashboard. 3. Click OK. 4. On any sheet, right-click an empty area of the Data pane at left, and select Create Calculated Field. 5. Give the calculation a descriptive name like Display sheet. In the formula text box, enter the name of the parameter you created above. Then click OK. 2816 Tableau Software Tableau Desktop and Web Authoring Help
Score: 0.8306743527803891
Source: tableau_desktop.pdf
Page: 1361
e. Click OK to return to the Calculated Field dialog box. 4. Repeat the previous step to create the following additional parameters: l Select Column 2 Heading l Select Row 1 Heading l Select Row 2 Heading Tip: Instead of typing each value in the list, click Add values from > Parameters to add them from Select Column 1 Heading. Create the calculated fields These steps use the Superstore sample to build the calculated fields that will take advantage of your parameters. 1. In the Data pane, click t
Score

In [ ]:
def answer_question(question):
    docs = retrieve(question)
    context='\n\n'.join([d.payload['content'] for d in docs])
    prompt=f'''You are a Tableau expert assistant. Use only the supplied context.\n\nContext:\n{context}\n\nQuestion:\n{question}'''
    response = groq_client.chat.completions.create(model='llama-3.3-70b-versatile', messages=[{'role':'user','content':prompt}], temperature=0)
    return response.choices[0].message.content, docs


In [ ]:
question=input('Ask a Tableau question: ')
answer, docs = answer_question(question)
print(answer)
for d in docs:
    print(d.payload['source'], d.payload['page'])


Ask a Tableau question: How do i create a calculated field ?
To create a calculated field in Tableau, follow these steps:

1. In a worksheet in Tableau, select **Analysis** > **Create Calculated Field**.
2. In the Calculation Editor that opens, give the calculated field a name.
3. In the Calculation Editor, enter a formula using a combination of functions, fields, and operators.
4. When finished, click **OK**.

Alternatively, you can also create a calculated field by:

1. Right-clicking an empty area of the Data pane at left and selecting **Create Calculated Field**.
2. Giving the calculation a descriptive name and entering a formula in the formula text box.
3. Clicking **OK** to save the calculated field.

The new calculated field will be added to the Data pane with an = in front of the data type icon to indicate it's a calculated field. You can then use the calculated field in your view.
tableau_desktop.pdf 3018
tableau_desktop.pdf 1361
tableau_desktop.pdf 2240
tableau_desktop.pdf 22

## Generate app.py

In [ ]:
%%writefile app.py

Overwriting app.py


In [ ]:
!find ./qdrant_data | head

./qdrant_data
./qdrant_data/.lock
./qdrant_data/collection
./qdrant_data/collection/tableau_docs
./qdrant_data/collection/tableau_docs/storage.sqlite
./qdrant_data/meta.json


In [ ]:
!zip -r qdrant_data.zip qdrant_data

  adding: qdrant_data/ (stored 0%)
  adding: qdrant_data/.lock (stored 0%)
  adding: qdrant_data/collection/ (stored 0%)
  adding: qdrant_data/collection/tableau_docs/ (stored 0%)
  adding: qdrant_data/collection/tableau_docs/storage.sqlite (deflated 55%)
  adding: qdrant_data/meta.json (deflated 57%)


In [ ]:
from google.colab import files
files.download("qdrant_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!du -sh qdrant_data

20M	qdrant_data
